# Dataset Split

Notebook này tạo **một lần duy nhất** ba split nội bộ từ **1000 ảnh train có ground-truth**:

- `train.csv`: 700 ảnh
- `valid.csv`: 100 ảnh
- `test.csv`: 200 ảnh internal test có mask

`data/test/test` của Kaggle **không dùng** cho benchmark nội bộ vì không có ground-truth mask.

Quy ước CSV:

- `image_id` là **tên file đầy đủ**, bao gồm extension, ví dụ `abc.jpeg`;
- image và mask được match bằng **tên file chính xác**, không chỉ bằng stem;
- path được lưu tương đối từ `data/`, ví dụ `train/train/abc.jpeg` và `train_gt/train_gt/abc.jpeg`;
- CSV không chứa project root, tiền tố `data/`, `foreground_pixels` hoặc `foreground_ratio`.

Notebook đọc mask RGB, xác định `neo_only`, `non_neo_only`, `both`, stratified split `700/100/200`, sau đó tạo hai labeled subset lồng nhau `175 ⊂ 350`.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        data_dir = candidate / "data"
        if (data_dir / "train" / "train").is_dir() and \
           (data_dir / "train_gt" / "train_gt").is_dir():
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy project chứa data/train/train và "
        f"data/train_gt/train_gt từ: {start}"
    )


try:
    PROJECT_ROOT = find_project_root(Path(__file__).resolve())
except NameError:
    PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
IMAGE_DIR = DATA_DIR / "train" / "train"
MASK_DIR = DATA_DIR / "train_gt" / "train_gt"
KAGGLE_TEST_DIR = DATA_DIR / "test" / "test"
SPLIT_DIR = DATA_DIR / "splits"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("DATA_DIR       :", DATA_DIR)
print("IMAGE_DIR      :", IMAGE_DIR)
print("MASK_DIR       :", MASK_DIR)
print("KAGGLE_TEST_DIR:", KAGGLE_TEST_DIR)
print("SPLIT_DIR      :", SPLIT_DIR)

PROJECT_ROOT   : B:\_projects\poply-image-segmentation
DATA_DIR       : B:\_projects\poply-image-segmentation\data
IMAGE_DIR      : B:\_projects\poply-image-segmentation\data\train\train
MASK_DIR       : B:\_projects\poply-image-segmentation\data\train_gt\train_gt
KAGGLE_TEST_DIR: B:\_projects\poply-image-segmentation\data\test\test
SPLIT_DIR      : B:\_projects\poply-image-segmentation\data\splits


In [2]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

BACKGROUND_RGB = (0, 0, 0)
NON_NEO_RGB = (0, 255, 0)
NEO_RGB = (255, 0, 0)

RGB_TO_CLASS = {
    BACKGROUND_RGB: 0,
    NON_NEO_RGB: 1,
    NEO_RGB: 2,
}

PALETTE = np.array(
    [BACKGROUND_RGB, NON_NEO_RGB, NEO_RGB],
    dtype=np.int32,
)

# Mask là JPEG nên một số pixel biên không đúng tuyệt đối ba màu chuẩn.
MAX_COLOR_DIST = 80
MAX_FAR_PIXEL_RATIO = 0.01

N_TOTAL_EXPECTED = 1000
N_TRAIN = 700
N_VALID = 100
N_TEST = 200
N_LABELED_1_4 = 175
N_LABELED_1_2 = 350

assert N_TRAIN + N_VALID + N_TEST == N_TOTAL_EXPECTED
assert N_LABELED_1_4 < N_LABELED_1_2 <= N_TRAIN

In [3]:
def list_images(folder: Path) -> list[Path]:
    if not folder.is_dir():
        raise FileNotFoundError(f"Không tìm thấy folder: {folder}")

    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def build_filename_map(files: list[Path]) -> dict[str, Path]:
    """Lập mapping bằng tên file đầy đủ, bao gồm extension."""
    result = {}

    for path in files:
        filename = path.name
        if filename in result:
            raise ValueError(
                f"Trùng tên file '{filename}' trong cùng folder:\n"
                f"- {result[filename]}\n"
                f"- {path}"
            )
        result[filename] = path

    return result


image_files = list_images(IMAGE_DIR)
mask_files = list_images(MASK_DIR)

image_map = build_filename_map(image_files)
mask_map = build_filename_map(mask_files)

image_names = set(image_map)
mask_names = set(mask_map)

missing_masks = sorted(image_names - mask_names)
missing_images = sorted(mask_names - image_names)

print(f"Images : {len(image_files)}")
print(f"Masks  : {len(mask_files)}")
print(f"Matched: {len(image_names & mask_names)}")

if missing_masks:
    print("Ảnh không có mask cùng tên:", missing_masks[:20])
if missing_images:
    print("Mask không có ảnh cùng tên:", missing_images[:20])

assert not missing_masks, "Có ảnh không có mask cùng tên chính xác."
assert not missing_images, "Có mask không có ảnh cùng tên chính xác."
assert len(image_names) == N_TOTAL_EXPECTED, (
    f"Kỳ vọng {N_TOTAL_EXPECTED} cặp, tìm thấy {len(image_names)}."
)

Images : 1000
Masks  : 1000
Matched: 1000


In [4]:
def inspect_mask(mask_path: Path) -> dict:
    """Đọc mask RGB và trả metadata gốc, không lưu các trường suy dẫn."""
    mask_rgb = np.asarray(Image.open(mask_path).convert("RGB"), dtype=np.uint8)
    h, w, channels = mask_rgb.shape
    assert channels == 3

    flat = mask_rgb.reshape(-1, 3).astype(np.int32)

    # Dùng int64 khi bình phương để tuyệt đối không tràn số.
    diff = flat[:, None, :] - PALETTE[None, :, :]
    d2 = np.sum(diff.astype(np.int64) ** 2, axis=2)
    cls = d2.argmin(axis=1)
    min_dist = np.sqrt(d2.min(axis=1))

    # Pixel JPEG mơ hồ, quá xa cả black/green/red, được coi là background.
    far_mask = min_dist > MAX_COLOR_DIST
    far_count = int(far_mask.sum())
    far_ratio = far_count / flat.shape[0]
    cls[far_mask] = RGB_TO_CLASS[BACKGROUND_RGB]

    if far_ratio > MAX_FAR_PIXEL_RATIO:
        raise ValueError(
            f"{mask_path.name}: {far_count}/{flat.shape[0]} pixel "
            f"({far_ratio:.2%}) quá xa palette; cần kiểm tra mask."
        )

    class_counts = np.bincount(cls, minlength=3)
    background_pixels = int(class_counts[0])
    non_neo_pixels = int(class_counts[1])
    neo_pixels = int(class_counts[2])

    has_non_neo = non_neo_pixels > 0
    has_neo = neo_pixels > 0

    if has_non_neo and has_neo:
        stratum = "both"
    elif has_non_neo:
        stratum = "non_neo_only"
    elif has_neo:
        stratum = "neo_only"
    else:
        stratum = "background_only"

    total_pixels = h * w
    assert background_pixels + non_neo_pixels + neo_pixels == total_pixels

    return {
        "width": w,
        "height": h,
        "stratum": stratum,
        "has_non_neoplastic": int(has_non_neo),
        "has_neoplastic": int(has_neo),
        "background_pixels": background_pixels,
        "non_neo_pixels": non_neo_pixels,
        "neo_pixels": neo_pixels,
    }


records = []

for idx, filename in enumerate(sorted(image_names), start=1):
    image_path = image_map[filename]
    mask_path = mask_map[filename]

    # Mapping bắt buộc bằng đúng tên file, bao gồm extension.
    assert image_path.name == mask_path.name == filename

    records.append({
        "image_id": filename,
        "image_path": image_path.relative_to(DATA_DIR).as_posix(),
        "mask_path": mask_path.relative_to(DATA_DIR).as_posix(),
        **inspect_mask(mask_path),
    })

    if idx % 100 == 0:
        print(f"Processed {idx}/{len(image_names)}")

dataset_df = pd.DataFrame(records)

assert dataset_df["image_id"].str.contains(r"\.[^.]+$", regex=True).all()
assert dataset_df["image_path"].str.startswith("train/train/").all()
assert dataset_df["mask_path"].str.startswith("train_gt/train_gt/").all()

display(dataset_df.head())
print("Shape:", dataset_df.shape)

Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000


,image_id,image_path,mask_path,width,height,stratum,has_non_neoplastic,has_neoplastic,background_pixels,non_neo_pixels,neo_pixels
0,0081835cf877e004e8bfb905b78a9139.jpeg,train/train/0081835cf877e004e8bfb905b78a9139.jpeg,train_gt/train_gt/0081835cf877e004e8bfb905b78a...,1280,995,neo_only,0,1,1252742,0,20858
1,00d827b8bcf15050fb9af0ef92846b60.jpeg,train/train/00d827b8bcf15050fb9af0ef92846b60.jpeg,train_gt/train_gt/00d827b8bcf15050fb9af0ef9284...,1280,995,neo_only,0,1,1197272,0,76328
2,00f9b6c911c4aa179344271fa54aa1b2.jpeg,train/train/00f9b6c911c4aa179344271fa54aa1b2.jpeg,train_gt/train_gt/00f9b6c911c4aa179344271fa54a...,1280,959,neo_only,0,1,1173318,0,54202
3,00fd197cd955fa095f978455cef3593c.jpeg,train/train/00fd197cd955fa095f978455cef3593c.jpeg,train_gt/train_gt/00fd197cd955fa095f978455cef3...,1280,977,non_neo_only,1,0,1143416,107144,0
4,01c13a3d810b000db55759486b8aa1b8.jpeg,train/train/01c13a3d810b000db55759486b8aa1b8.jpeg,train_gt/train_gt/01c13a3d810b000db55759486b8a...,1280,995,non_neo_only,1,0,1244543,29057,0


Shape: (1000, 11)


In [5]:
# Các trường foreground chỉ được tính tạm để thống kê, không ghi vào CSV.
stats_source = dataset_df.assign(
    foreground_pixels=(
        dataset_df["non_neo_pixels"] + dataset_df["neo_pixels"]
    ),
    foreground_ratio=(
        (dataset_df["non_neo_pixels"] + dataset_df["neo_pixels"])
        / (dataset_df["width"] * dataset_df["height"])
    ),
)

stratum_stats = (
    stats_source
    .groupby("stratum", dropna=False)
    .agg(
        images=("image_id", "count"),
        mean_foreground_ratio=("foreground_ratio", "mean"),
        total_non_neo_pixels=("non_neo_pixels", "sum"),
        total_neo_pixels=("neo_pixels", "sum"),
    )
    .reset_index()
)

stratum_stats["image_ratio"] = stratum_stats["images"] / len(dataset_df)
display(stratum_stats)

background_only_count = int(
    (dataset_df["stratum"] == "background_only").sum()
)

if background_only_count:
    print(f"WARNING: phát hiện {background_only_count} mask background_only.")
else:
    print("OK: không phát hiện mask background_only.")

,stratum,images,mean_foreground_ratio,total_non_neo_pixels,total_neo_pixels,image_ratio
0,both,49,0.059815,589238,3116468,0.049
1,neo_only,694,0.062302,0,54349685,0.694
2,non_neo_only,257,0.023930,7732028,0,0.257


OK: không phát hiện mask background_only.


In [6]:
# --------------------------------------------------------------
# Stratified split:
# 1000 -> 800 train+valid + 200 test
# 800  -> 700 train       + 100 valid
# --------------------------------------------------------------

train_valid_df, test_df = train_test_split(
    dataset_df,
    test_size=N_TEST,
    random_state=SEED,
    shuffle=True,
    stratify=dataset_df["stratum"],
)

train_df, valid_df = train_test_split(
    train_valid_df,
    test_size=N_VALID,
    random_state=SEED,
    shuffle=True,
    stratify=train_valid_df["stratum"],
)

train_df = train_df.copy().reset_index(drop=True)
valid_df = valid_df.copy().reset_index(drop=True)
test_df = test_df.copy().reset_index(drop=True)

assert len(train_df) == 700
assert len(valid_df) == 100
assert len(test_df) == 200

assert set(train_df.image_id).isdisjoint(valid_df.image_id)
assert set(train_df.image_id).isdisjoint(test_df.image_id)
assert set(valid_df.image_id).isdisjoint(test_df.image_id)

print("Train:", len(train_df))
print("Valid:", len(valid_df))
print("Test :", len(test_df))

Train: 700
Valid: 100
Test : 200


In [7]:
# --------------------------------------------------------------
# Nested labeled subsets trong train:
# labeled_1_4 (175) subset của labeled_1_2 (350)
# --------------------------------------------------------------

train_df["labeled_1_4"] = 0
train_df["labeled_1_2"] = 0

labeled_1_2_df, _ = train_test_split(
    train_df,
    train_size=N_LABELED_1_2,
    random_state=SEED,
    shuffle=True,
    stratify=train_df["stratum"],
)

labeled_1_4_df, _ = train_test_split(
    labeled_1_2_df,
    train_size=N_LABELED_1_4,
    random_state=SEED,
    shuffle=True,
    stratify=labeled_1_2_df["stratum"],
)

ids_1_2 = set(labeled_1_2_df["image_id"])
ids_1_4 = set(labeled_1_4_df["image_id"])

train_df.loc[
    train_df["image_id"].isin(ids_1_2),
    "labeled_1_2"
] = 1

train_df.loc[
    train_df["image_id"].isin(ids_1_4),
    "labeled_1_4"
] = 1

assert int(train_df["labeled_1_4"].sum()) == 175
assert int(train_df["labeled_1_2"].sum()) == 350
assert ids_1_4.issubset(ids_1_2)

print("1/4 labeled  :", int(train_df["labeled_1_4"].sum()))
print("1/4 unlabeled:", int((train_df["labeled_1_4"] == 0).sum()))
print("1/2 labeled  :", int(train_df["labeled_1_2"].sum()))
print("1/2 unlabeled:", int((train_df["labeled_1_2"] == 0).sum()))
print("Nested subset:", ids_1_4.issubset(ids_1_2))

1/4 labeled  : 175
1/4 unlabeled: 525
1/2 labeled  : 350
1/2 unlabeled: 350
Nested subset: True


In [8]:
def split_distribution(df, split_name):
    counts = (
        df["stratum"]
        .value_counts()
        .rename_axis("stratum")
        .reset_index(name="count")
    )

    counts["ratio"] = counts["count"] / len(df)
    counts.insert(0, "split", split_name)

    return counts


split_stats_df = pd.concat(
    [
        split_distribution(train_df, "train"),
        split_distribution(valid_df, "valid"),
        split_distribution(test_df, "test"),
    ],
    ignore_index=True,
)

display(split_stats_df)

,split,stratum,count,ratio
0,train,neo_only,486,0.694286
1,train,non_neo_only,180,0.257143
2,train,both,34,0.048571
3,valid,neo_only,69,0.690000
4,valid,non_neo_only,26,0.260000
5,valid,both,5,0.050000
6,test,neo_only,139,0.695000
7,test,non_neo_only,51,0.255000
8,test,both,10,0.050000


In [9]:
def label_ratio_stats(train_df, column, ratio_name):
    rows = []

    for status, group in train_df.groupby(column):
        label_status = "labeled" if status == 1 else "unlabeled"
        counts = group["stratum"].value_counts()

        rows.append({
            "label_ratio": ratio_name,
            "status": label_status,
            "total": len(group),
            "neo_only": int(counts.get("neo_only", 0)),
            "non_neo_only": int(counts.get("non_neo_only", 0)),
            "both": int(counts.get("both", 0)),
            "background_only": int(counts.get("background_only", 0)),
        })

    return pd.DataFrame(rows)


label_stats_df = pd.concat(
    [
        label_ratio_stats(train_df, "labeled_1_4", "1/4"),
        label_ratio_stats(train_df, "labeled_1_2", "1/2"),
    ],
    ignore_index=True,
)

display(label_stats_df)

,label_ratio,status,total,neo_only,non_neo_only,both,background_only
0,1/4,unlabeled,525,364,135,26,0
1,1/4,labeled,175,122,45,8,0
2,1/2,unlabeled,350,243,90,17,0
3,1/2,labeled,350,243,90,17,0


In [10]:
def pixel_stats(df, split_name):
    bg = int(df["background_pixels"].sum())
    non_neo = int(df["non_neo_pixels"].sum())
    neo = int(df["neo_pixels"].sum())

    total = bg + non_neo + neo

    return {
        "split": split_name,
        "images": len(df),
        "background_pixels": bg,
        "non_neo_pixels": non_neo,
        "neo_pixels": neo,
        "background_ratio": bg / total,
        "non_neo_ratio": non_neo / total,
        "neo_ratio": neo / total,
    }


pixel_stats_df = pd.DataFrame([
    pixel_stats(train_df, "train"),
    pixel_stats(valid_df, "valid"),
    pixel_stats(test_df, "test"),
])

display(pixel_stats_df)

,split,images,background_pixels,non_neo_pixels,neo_pixels,background_ratio,non_neo_ratio,neo_ratio
0,train,700,835129817,5655991,40030832,0.948131,0.006421,0.045447
1,valid,100,119740948,1269472,4744460,0.952177,0.010095,0.037728
2,test,200,237505016,1395803,12690861,0.944010,0.005548,0.050442


In [11]:
base_columns = [
    "image_id",
    "image_path",
    "mask_path",
    "stratum",
    "has_non_neoplastic",
    "has_neoplastic",
    "width",
    "height",
    "background_pixels",
    "non_neo_pixels",
    "neo_pixels",
]

train_columns = base_columns + ["labeled_1_4", "labeled_1_2"]

train_csv = SPLIT_DIR / "train.csv"
valid_csv = SPLIT_DIR / "valid.csv"
test_csv = SPLIT_DIR / "test.csv"

train_out = train_df[train_columns].sort_values("image_id").reset_index(drop=True)
valid_out = valid_df[base_columns].sort_values("image_id").reset_index(drop=True)
test_out = test_df[base_columns].sort_values("image_id").reset_index(drop=True)

train_out.to_csv(train_csv, index=False)
valid_out.to_csv(valid_csv, index=False)
test_out.to_csv(test_csv, index=False)

print("Saved:")
print(train_csv)
print(valid_csv)
print(test_csv)

Saved:
B:\_projects\poply-image-segmentation\data\splits\train.csv
B:\_projects\poply-image-segmentation\data\splits\valid.csv
B:\_projects\poply-image-segmentation\data\splits\test.csv


In [12]:
# Đọc lại từ disk và validation toàn diện schema/mapping/path/split.
check_splits = {
    "train": pd.read_csv(SPLIT_DIR / "train.csv"),
    "valid": pd.read_csv(SPLIT_DIR / "valid.csv"),
    "test": pd.read_csv(SPLIT_DIR / "test.csv"),
}

assert {name: len(df) for name, df in check_splits.items()} == {
    "train": N_TRAIN,
    "valid": N_VALID,
    "test": N_TEST,
}

assert list(check_splits["train"].columns) == train_columns
assert list(check_splits["valid"].columns) == base_columns
assert list(check_splits["test"].columns) == base_columns

for split_name, df in check_splits.items():
    assert not df["image_id"].duplicated().any()
    assert not df["image_path"].duplicated().any()
    assert not df["mask_path"].duplicated().any()

    for row in df.itertuples(index=False):
        image_rel = Path(row.image_path)
        mask_rel = Path(row.mask_path)

        assert not image_rel.is_absolute()
        assert not mask_rel.is_absolute()
        assert image_rel.parts[:2] == ("train", "train")
        assert mask_rel.parts[:2] == ("train_gt", "train_gt")
        assert image_rel.name == mask_rel.name == row.image_id
        assert (DATA_DIR / image_rel).is_file()
        assert (DATA_DIR / mask_rel).is_file()

        total_pixels = row.width * row.height
        class_pixels = (
            row.background_pixels + row.non_neo_pixels + row.neo_pixels
        )
        assert class_pixels == total_pixels
        assert row.has_non_neoplastic == int(row.non_neo_pixels > 0)
        assert row.has_neoplastic == int(row.neo_pixels > 0)

all_ids = [set(df["image_id"]) for df in check_splits.values()]
assert all_ids[0].isdisjoint(all_ids[1])
assert all_ids[0].isdisjoint(all_ids[2])
assert all_ids[1].isdisjoint(all_ids[2])

check_train = check_splits["train"]
assert int(check_train["labeled_1_4"].sum()) == N_LABELED_1_4
assert int(check_train["labeled_1_2"].sum()) == N_LABELED_1_2
assert check_train["labeled_1_4"].isin([0, 1]).all()
assert check_train["labeled_1_2"].isin([0, 1]).all()

ids_14 = set(check_train.loc[check_train["labeled_1_4"] == 1, "image_id"])
ids_12 = set(check_train.loc[check_train["labeled_1_2"] == 1, "image_id"])
assert ids_14.issubset(ids_12)

print("ALL CHECKS PASSED")
for split_name, df in check_splits.items():
    print(f"\n{split_name}.csv")
    display(df.head())

ALL CHECKS PASSED

train.csv


,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels,labeled_1_4,labeled_1_2
0,0081835cf877e004e8bfb905b78a9139.jpeg,train/train/0081835cf877e004e8bfb905b78a9139.jpeg,train_gt/train_gt/0081835cf877e004e8bfb905b78a...,neo_only,0,1,1280,995,1252742,0,20858,0,1
1,00f9b6c911c4aa179344271fa54aa1b2.jpeg,train/train/00f9b6c911c4aa179344271fa54aa1b2.jpeg,train_gt/train_gt/00f9b6c911c4aa179344271fa54a...,neo_only,0,1,1280,959,1173318,0,54202,1,1
2,00fd197cd955fa095f978455cef3593c.jpeg,train/train/00fd197cd955fa095f978455cef3593c.jpeg,train_gt/train_gt/00fd197cd955fa095f978455cef3...,non_neo_only,1,0,1280,977,1143416,107144,0,0,0
3,01c13a3d810b000db55759486b8aa1b8.jpeg,train/train/01c13a3d810b000db55759486b8aa1b8.jpeg,train_gt/train_gt/01c13a3d810b000db55759486b8a...,non_neo_only,1,0,1280,995,1244543,29057,0,0,1
4,01e80947f9c02314fc58bdb26679bff5.jpeg,train/train/01e80947f9c02314fc58bdb26679bff5.jpeg,train_gt/train_gt/01e80947f9c02314fc58bdb26679...,neo_only,0,1,1280,995,1161474,0,112126,0,1



valid.csv


,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels
0,0496b83ef461c2a337948a41964c1d4f.jpeg,train/train/0496b83ef461c2a337948a41964c1d4f.jpeg,train_gt/train_gt/0496b83ef461c2a337948a41964c...,non_neo_only,1,0,1280,995,1237881,35719,0
1,051fc90e85d064f787d4be168d951058.jpeg,train/train/051fc90e85d064f787d4be168d951058.jpeg,train_gt/train_gt/051fc90e85d064f787d4be168d95...,neo_only,0,1,1280,977,1225389,0,25171
2,07907941a8490fd7df8e26031fbb5e52.jpeg,train/train/07907941a8490fd7df8e26031fbb5e52.jpeg,train_gt/train_gt/07907941a8490fd7df8e26031fbb...,neo_only,0,1,1280,995,1256778,0,16822
3,0bbd4ce2a1405f7d507592b964b7955a.jpeg,train/train/0bbd4ce2a1405f7d507592b964b7955a.jpeg,train_gt/train_gt/0bbd4ce2a1405f7d507592b964b7...,neo_only,0,1,1280,995,1233760,0,39840
4,0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,train/train/0c90be319e1d6d19a707a4fdcc58c9c9.jpeg,train_gt/train_gt/0c90be319e1d6d19a707a4fdcc58...,non_neo_only,1,0,1280,977,1225582,24978,0



test.csv


,image_id,image_path,mask_path,stratum,has_non_neoplastic,has_neoplastic,width,height,background_pixels,non_neo_pixels,neo_pixels
0,00d827b8bcf15050fb9af0ef92846b60.jpeg,train/train/00d827b8bcf15050fb9af0ef92846b60.jpeg,train_gt/train_gt/00d827b8bcf15050fb9af0ef9284...,neo_only,0,1,1280,995,1197272,0,76328
1,01ff069910ea2abe3c84417fda801941.jpeg,train/train/01ff069910ea2abe3c84417fda801941.jpeg,train_gt/train_gt/01ff069910ea2abe3c84417fda80...,neo_only,0,1,1280,995,1261623,0,11977
2,040781d54ffd6a8ee924dbd70f618d52.jpeg,train/train/040781d54ffd6a8ee924dbd70f618d52.jpeg,train_gt/train_gt/040781d54ffd6a8ee924dbd70f61...,neo_only,0,1,1280,995,1212723,0,60877
3,081835cf877e004e8bfb905b78a91391.jpeg,train/train/081835cf877e004e8bfb905b78a91391.jpeg,train_gt/train_gt/081835cf877e004e8bfb905b78a9...,neo_only,0,1,1280,995,1237606,0,35994
4,099130d98309817287a8aef566999467.jpeg,train/train/099130d98309817287a8aef566999467.jpeg,train_gt/train_gt/099130d98309817287a8aef56699...,neo_only,0,1,1280,995,1197251,0,76349


## Quy ước dùng về sau

- `image_id` là tên file chính xác, bao gồm extension.
- `image_path` và `mask_path` là path tương đối từ thư mục `data/`; khi đọc file dùng `DATA_DIR / csv_path`.
- `train.csv` vẫn chứa `mask_path` của toàn bộ 700 ảnh vì dữ liệu nguồn đều có GT.
- Với cấu hình `1/4`, row có `labeled_1_4 == 0` phải được coi là unlabeled.
- Với cấu hình `1/2`, row có `labeled_1_2 == 0` phải được coi là unlabeled.
- `Dataset`/`DataLoader` không được trả ground-truth mask của phần unlabeled cho training loop.